# Module Overview

This module covers everything you need to know about parsing and ingesting data for RAG systems, from basics text file to complex PDFs and databases. We'll use langchain and explore each technique with practical examples

Table of Contents

- Introduction to Data Ingestion
- Text Files (.txt)
- Markdown Files (.md)
- PDF Documents
- Microsoft Word Documents
- CSV and Excel Files
- JSON and Structured Data
- Web Scraping
- Databases (SQL)
- Audio and Video Transcripts
- Advanced Techniques
- Best Practices

### Introduction to Data Ingestion

In [1]:
import os
from typing import Dict, List, Any
import pandas as pd

In [2]:
from langchain_core.documents import Document
from langchain_text_splitters import(
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)

print("Setup Complete!!")

e:\RAG Bootcamp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup Complete!!


### Understanding of Document Structure in Langchain

In [3]:
## Create a simple document
doc = Document(
    page_content="This is a main text content that will be embedded and searched",
    metadata={
        "source": "example.txt",
        "page": 1,
        "author": "Manish Bhoi",
        "created_date": "2026-04-17"
    }
)

print("Document Structure")

print(f"Content :{doc.page_content}")
print(f"Metadata :{doc.metadata}")

## Why metadata matters :
print("\n Metadata is crusial for :")
print("- Filtering search results")
print("- Tracking document sources")
print("- Providing Context in responses")
print("- Debugging and Audinting")

Document Structure
Content :This is a main text content that will be embedded and searched
Metadata :{'source': 'example.txt', 'page': 1, 'author': 'Manish Bhoi', 'created_date': '2026-04-17'}

 Metadata is crusial for :
- Filtering search results
- Tracking document sources
- Providing Context in responses
- Debugging and Audinting


### Text Files (.txt) - The Simplest Case 

In [4]:
## Create a simple txt file
import os
os.makedirs("data/text_files", exist_ok=True)

In [5]:
sample_texts = {
    "data/text_files/Python_intro.txt": """Python Introduction  
    
    Python is a high-level, interpreted programming language known for its simple syntax, readability, and versatility. It is widely used for software development, web applications, data science, automation, and artificial intelligence.
    
    Key Features
        Easy to learn and use
        Simple and readable syntax
        Interpreted language
        Object-oriented programming support
        Large standard library
        Cross-platform compatibility
        Open-source and free
        Strong community support
        Supports multiple programming paradigms
        
    Ideal for beginners
    Faster development process
    Used in data science and AI
    Excellent libraries and frameworks
    High demand in the job market
    Suitable for automation and scripting
    Works on multiple operating systems
    """,

    "data/text_files/Machine_Learning_Basics.txt": """Basics of ML
    
    Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn from data and improve performance without being explicitly programmed.


    Key Features
        Learns from data automatically
        Improves with experience
        Identifies patterns and trends
        Makes predictions and decisions
        Handles large amounts of data
        Supports automation
        Adapts to new data over time

    Helps in accurate predictions
    Automates repetitive tasks
    Improves decision-making
    Used in recommendation systems
    Detects fraud and spam
    Powers chatbots and virtual assistants
    Essential for modern AI applications
    """
}

for filepath,content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample txt file created")

Sample txt file created


### TextLoader - Read Single File

In [6]:
from langchain_community.document_loaders import TextLoader

## Loading Single Text File
loader = TextLoader("data/text_files/python_intro.txt", encoding="utf-8")

documents = loader.load()
print(f"Loaded {len(documents)} documents")
print(f"Content Preview : {documents[0].page_content[:100]}")
print(f"Metadata : {documents[0].metadata}")

Loaded 1 documents
Content Preview : Python Introduction  

    Python is a high-level, interpreted programming language known for its si
Metadata : {'source': 'data/text_files/python_intro.txt'}


### DirectoryLoader - Multiple Text Files

In [7]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

## Load all Text Files from Directory
dir_loader = DirectoryLoader(
    "data/text_files",
    glob="**/*.txt", ## Patterns to match files
    loader_cls= TextLoader, ## Loader class to use
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)

documents = dir_loader.load()

print(f"Loader {len(documents)} Documents")
for i, doc in enumerate(documents):
    print(f"\nDocument {i+1}: ")
    print(f" Source : {doc.metadata['source']}")
    print(f" Length : {len(doc.page_content)} characters")


# Analysis 
print("\n DirectoryLoader Characteristics:")
print("Advantages: ")
print("- Loads Multiple files at once")
print("- Supports glob patterns")
print("- Progress Tracking")
print("- Recursive directory Scanning")

print("\nDisadvantages: ")
print("- All files must be same type")
print("- Limited error handling per file")
print("- Can be memory intensive for large directories")

100%|██████████| 2/2 [00:00<00:00, 262.73it/s]

Loader 2 Documents

Document 1: 
 Source : data\text_files\Machine_Learning_Basics.txt
 Length : 705 characters

Document 2: 
 Source : data\text_files\Python_intro.txt
 Length : 841 characters

 DirectoryLoader Characteristics:
Advantages: 
- Loads Multiple files at once
- Supports glob patterns
- Progress Tracking
- Recursive directory Scanning

Disadvantages: 
- All files must be same type
- Limited error handling per file
- Can be memory intensive for large directories


### Text Splitting Strategies

In [10]:
## Different text splitting strategies
from langchain_text_splitters import(
    RecursiveCharacterTextSplitter, 
    CharacterTextSplitter, 
    TokenTextSplitter
)

print(documents)

text = documents[0].page_content
text

[Document(metadata={'source': 'data\\text_files\\Machine_Learning_Basics.txt'}, page_content='Basics of ML\n\n    Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn from data and improve performance without being explicitly programmed.\n\n\n    Key Features\n        Learns from data automatically\n        Improves with experience\n        Identifies patterns and trends\n        Makes predictions and decisions\n        Handles large amounts of data\n        Supports automation\n        Adapts to new data over time\n\n    Helps in accurate predictions\n    Automates repetitive tasks\n    Improves decision-making\n    Used in recommendation systems\n    Detects fraud and spam\n    Powers chatbots and virtual assistants\n    Essential for modern AI applications\n    '), Document(metadata={'source': 'data\\text_files\\Python_intro.txt'}, page_content='Python Introduction  \n\n    Python is a high-level, interpreted programming language known fo

'Basics of ML\n\n    Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn from data and improve performance without being explicitly programmed.\n\n\n    Key Features\n        Learns from data automatically\n        Improves with experience\n        Identifies patterns and trends\n        Makes predictions and decisions\n        Handles large amounts of data\n        Supports automation\n        Adapts to new data over time\n\n    Helps in accurate predictions\n    Automates repetitive tasks\n    Improves decision-making\n    Used in recommendation systems\n    Detects fraud and spam\n    Powers chatbots and virtual assistants\n    Essential for modern AI applications\n    '

In [ ]:
### Method 1 - CharacterTextSplitter - Split by Character

print("=========CHARACTER TEXT SPLITTER========")
char_splitter = CharacterTextSplitter(
    chunk_size=200, # Number of characters per chunk
    chunk_overlap=20, # Overlap between chunks
    separator='\n', # Separator between chunks
    length_function=len # Function to calculate length
)

chunks = char_splitter.split_text(text)
print(f'Create {len(chunks)} chunks')
print(f'First chunk : {chunks[0][:100]}')


=========CHARACTER TEXT SPLITTER========
Create 4 chunks
First chunk : Basics of ML
    Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables comp


In [15]:
print(chunks[0])
print("======================")
print(chunks[1])
print("======================")
print(chunks[2])
print("======================")
print(chunks[3])

Basics of ML
    Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn from data and improve performance without being explicitly programmed.
Key Features
        Learns from data automatically
        Improves with experience
        Identifies patterns and trends
        Makes predictions and decisions
Handles large amounts of data
        Supports automation
        Adapts to new data over time
    Helps in accurate predictions
    Automates repetitive tasks
    Improves decision-making
Used in recommendation systems
    Detects fraud and spam
    Powers chatbots and virtual assistants
    Essential for modern AI applications


In [19]:
## Method 2 - RecursiveCharacterSplitter - Split by Character (RECOMMENDED)

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, # Number of characters per chunk
    chunk_overlap=20, # Overlap between chunks
    separators=['\n\n', '\n', " ", ""], # Separator between chunks 
    length_function=len # Function to calculate length
)


recursive_chunks = recursive_splitter.split_text(text)
print(f'Create {len(recursive_chunks)} chunks')
print(f'First chunk : {recursive_chunks[0][:100]}')

Create 5 chunks
First chunk : Basics of ML

    Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables com


In [20]:
print(recursive_chunks[0])
print("======================")
print(recursive_chunks[1])
print("======================")
print(recursive_chunks[2])
print("======================")
print(recursive_chunks[3])
print("======================")
print(recursive_chunks[4])
print("======================")

Basics of ML

    Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn from data and improve performance without being explicitly programmed.
Key Features
        Learns from data automatically
        Improves with experience
        Identifies patterns and trends
        Makes predictions and decisions
Handles large amounts of data
        Supports automation
        Adapts to new data over time
Helps in accurate predictions
    Automates repetitive tasks
    Improves decision-making
    Used in recommendation systems
    Detects fraud and spam
    Powers chatbots and virtual assistants
Essential for modern AI applications


In [21]:
## Method 3 - TokenTextSplitter - Split by Token (RECOMMENDED)

token_splitter = TokenTextSplitter(
    chunk_size=50, # Number of tokens per chunk
    chunk_overlap=10, # Overlap between chunks\
    length_function=len # Function to calculate length
)

token_chunks = token_splitter.split_text(text)
print(f'Create {len(token_chunks)} chunks')
print(f'First chunk : {token_chunks[0][:100]}')

Create 5 chunks
First chunk : Basics of ML

    Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables com


In [22]:
print(token_chunks[0])
print("======================")
print(token_chunks[1])
print("======================")
print(token_chunks[2])
print("======================")
print(token_chunks[3])
print("======================")
print(token_chunks[4])
print("======================")

Basics of ML

    Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn from data and improve performance without being explicitly programmed.


    Key Features
     
   Key Features
        Learns from data automatically
        Improves with experience
        Identifies patterns and trends
        Makes
 trends
        Makes predictions and decisions
        Handles large amounts of data
        Supports automation
        Adapts to new data
      Adapts to new data over time

    Helps in accurate predictions
    Automates repetitive tasks
    Improves decision-making
    Used in recommendation systems
   
   Used in recommendation systems
    Detects fraud and spam
    Powers chatbots and virtual assistants
    Essential for modern AI applications
    
